# Traffic Demand Dataset Preparation

This notebook performs a complete data audit, cleaning, preprocessing, exploratory data analysis, and feature engineering workflow for traffic demand prediction. The target variable is `demand`.

The final output is a training-ready feature matrix for train/test plus saved processed CSV files.


## 1. Setup

Load libraries, configure plotting, and define reusable helpers.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import LabelEncoder

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 200)
sns.set_theme(style='whitegrid', palette='Set2')

DATA_DIR = Path('.')
TRAIN_PATH = DATA_DIR / 'train.csv'
TEST_PATH = DATA_DIR / 'test.csv'
SAMPLE_SUBMISSION_PATH = DATA_DIR / 'sample_submission.csv'

TARGET = 'demand'
ID_COL = 'Index'
RANDOM_STATE = 42


## 2. Load Data

Read the original files and keep untouched raw copies for comparison.


In [ ]:
train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

print(f'Train shape: {train_raw.shape}')
print(f'Test shape : {test_raw.shape}')
print(f'Sample submission shape: {sample_submission.shape}')

display(train_raw.head())
display(test_raw.head())


## 3. Data Audit

Check schema, missing values, duplicates, target quality, and train/test differences before changing anything.


In [ ]:
def dataset_audit(df, name):
    audit = pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'missing_count': df.isna().sum(),
        'missing_pct': (df.isna().mean() * 100).round(2),
        'unique_count': df.nunique(dropna=True),
    })
    print(f'===== {name} audit =====')
    print(f'Rows: {len(df):,} | Columns: {df.shape[1]:,} | Duplicate rows: {df.duplicated().sum():,}')
    display(audit)


dataset_audit(train_raw, 'Train')
dataset_audit(test_raw, 'Test')

print('Target summary')
display(train_raw[TARGET].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).to_frame())

print('Target missing:', train_raw[TARGET].isna().sum())
print('Target outside [0, 1]:', ((train_raw[TARGET] < 0) | (train_raw[TARGET] > 1)).sum())
print('Unseen geohashes in test:', len(set(test_raw['geohash']) - set(train_raw['geohash'])))


In [ ]:
categorical_audit_cols = ['RoadType', 'LargeVehicles', 'Landmarks', 'Weather', 'NumberofLanes']
for col in categorical_audit_cols:
    print(f'\n===== {col} =====')
    levels = pd.concat([
        train_raw[col].value_counts(dropna=False, normalize=True).rename('train_pct'),
        test_raw[col].value_counts(dropna=False, normalize=True).rename('test_pct')
    ], axis=1).fillna(0).sort_values('train_pct', ascending=False)
    display((levels * 100).round(2))


## 4. Cleaning Strategy

Cleaning choices:

- Validate and parse timestamps into hour/minute/time-slot features.
- Preserve missingness indicators for columns where absence may carry signal.
- Fill categorical gaps with `Unknown`.
- Impute `Temperature` using train-derived medians by `Weather` and `hour`, then global train median fallback.
- Keep the target unchanged because `demand` is valid and bounded; create a log target only as an optional modeling helper.
- Avoid target leakage by calculating target encodings out-of-fold for train rows.


In [ ]:
CAT_COLS = ['geohash', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather']
BASE_NUM_COLS = ['day', 'NumberofLanes', 'Temperature']


def parse_timestamp_features(df):
    out = df.copy()
    time_parts = out['timestamp'].astype(str).str.split(':', expand=True)
    out['hour'] = pd.to_numeric(time_parts[0], errors='coerce').astype('Int64')
    out['minute'] = pd.to_numeric(time_parts[1], errors='coerce').astype('Int64')
    out['timestamp_invalid'] = (out['hour'].isna() | out['minute'].isna()).astype(int)

    out['hour'] = out['hour'].fillna(0).astype(int)
    out['minute'] = out['minute'].fillna(0).astype(int)
    out['time_index'] = out['hour'] * 4 + (out['minute'] // 15)
    out['total_minutes'] = out['hour'] * 60 + out['minute']

    out['hour_sin'] = np.sin(2 * np.pi * out['hour'] / 24)
    out['hour_cos'] = np.cos(2 * np.pi * out['hour'] / 24)
    out['time_index_sin'] = np.sin(2 * np.pi * out['time_index'] / 96)
    out['time_index_cos'] = np.cos(2 * np.pi * out['time_index'] / 96)

    out['is_morning_peak'] = out['hour'].between(7, 10).astype(int)
    out['is_evening_peak'] = out['hour'].between(16, 19).astype(int)
    out['is_late_night'] = out['hour'].between(0, 5).astype(int)
    out['is_peak_hour'] = ((out['is_morning_peak'] == 1) | (out['is_evening_peak'] == 1)).astype(int)
    return out


def decode_geohash(value):
    """Decode a geohash string to approximate latitude/longitude without external packages."""
    if pd.isna(value):
        return np.nan, np.nan
    base32 = '0123456789bcdefghjkmnpqrstuvwxyz'
    bits = [16, 8, 4, 2, 1]
    lat = [-90.0, 90.0]
    lon = [-180.0, 180.0]
    even = True
    try:
        for char in str(value).lower():
            cd = base32.index(char)
            for mask in bits:
                if even:
                    mid = (lon[0] + lon[1]) / 2
                    if cd & mask:
                        lon[0] = mid
                    else:
                        lon[1] = mid
                else:
                    mid = (lat[0] + lat[1]) / 2
                    if cd & mask:
                        lat[0] = mid
                    else:
                        lat[1] = mid
                even = not even
        return (lat[0] + lat[1]) / 2, (lon[0] + lon[1]) / 2
    except ValueError:
        return np.nan, np.nan


def add_geohash_features(df):
    out = df.copy()
    out['geohash'] = out['geohash'].astype(str)
    out['geohash_prefix_3'] = out['geohash'].str[:3]
    out['geohash_prefix_4'] = out['geohash'].str[:4]
    out['geohash_prefix_5'] = out['geohash'].str[:5]

    unique_geo = pd.DataFrame({'geohash': out['geohash'].drop_duplicates()})
    unique_geo[['geo_lat', 'geo_lon']] = pd.DataFrame(
        unique_geo['geohash'].map(decode_geohash).tolist(), index=unique_geo.index
    )
    out = out.merge(unique_geo, on='geohash', how='left')
    return out


def clean_base(train_df, test_df):
    train = train_df.copy()
    test = test_df.copy()
    train['is_train'] = 1
    test['is_train'] = 0
    test[TARGET] = np.nan
    combined = pd.concat([train, test], axis=0, ignore_index=True)

    combined = parse_timestamp_features(combined)
    combined = add_geohash_features(combined)

    for col in ['RoadType', 'Weather', 'Temperature']:
        combined[f'{col}_was_missing'] = combined[col].isna().astype(int)

    for col in ['RoadType', 'LargeVehicles', 'Landmarks', 'Weather']:
        combined[col] = combined[col].fillna('Unknown').astype(str).str.strip()

    combined['NumberofLanes'] = pd.to_numeric(combined['NumberofLanes'], errors='coerce')
    lane_median = train_df['NumberofLanes'].median()
    combined['NumberofLanes'] = combined['NumberofLanes'].fillna(lane_median).astype(int)
    combined['has_many_lanes'] = (combined['NumberofLanes'] >= 4).astype(int)

    combined['Temperature'] = pd.to_numeric(combined['Temperature'], errors='coerce')
    train_part = combined[combined['is_train'] == 1].copy()
    temp_by_weather_hour = train_part.groupby(['Weather', 'hour'])['Temperature'].median()
    temp_by_weather = train_part.groupby('Weather')['Temperature'].median()
    global_temp_median = train_part['Temperature'].median()

    def impute_temp(row):
        if pd.notna(row['Temperature']):
            return row['Temperature']
        key = (row['Weather'], row['hour'])
        if key in temp_by_weather_hour.index and pd.notna(temp_by_weather_hour.loc[key]):
            return temp_by_weather_hour.loc[key]
        if row['Weather'] in temp_by_weather.index and pd.notna(temp_by_weather.loc[row['Weather']]):
            return temp_by_weather.loc[row['Weather']]
        return global_temp_median

    combined['Temperature'] = combined.apply(impute_temp, axis=1)
    combined['temp_capped'] = combined['Temperature'].clip(
        train_part['Temperature'].quantile(0.01),
        train_part['Temperature'].quantile(0.99)
    )

    combined['large_vehicle_allowed'] = (combined['LargeVehicles'].str.lower() == 'allowed').astype(int)
    combined['has_landmark'] = (combined['Landmarks'].str.lower() == 'yes').astype(int)
    combined['road_weather'] = combined['RoadType'] + '_' + combined['Weather']
    combined['lane_vehicle_interaction'] = combined['NumberofLanes'] * combined['large_vehicle_allowed']
    combined['peak_lane_interaction'] = combined['is_peak_hour'] * combined['NumberofLanes']

    train_clean = combined[combined['is_train'] == 1].drop(columns=['is_train']).copy()
    test_clean = combined[combined['is_train'] == 0].drop(columns=['is_train', TARGET]).copy()
    return train_clean, test_clean


train_clean, test_clean = clean_base(train_raw, test_raw)

print('Remaining missing values in train:')
display(train_clean.isna().sum().sort_values(ascending=False).head(15).to_frame('missing_count'))
print('Remaining missing values in test:')
display(test_clean.isna().sum().sort_values(ascending=False).head(15).to_frame('missing_count'))

display(train_clean.head())


## 5. Strong EDA

Explore the target distribution, temporal patterns, categorical effects, geography, and train/test stability.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

sns.histplot(train_clean[TARGET], bins=60, kde=True, ax=axes[0])
axes[0].set_title('Demand distribution')

sns.boxplot(x=train_clean[TARGET], ax=axes[1])
axes[1].set_title('Demand spread and high-demand tail')

sns.histplot(np.log1p(train_clean[TARGET]), bins=60, kde=True, ax=axes[2])
axes[2].set_title('log1p(demand) distribution')

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(20, 12))

time_demand = train_clean.groupby('time_index')[TARGET].agg(['mean', 'median', 'count']).reset_index()
sns.lineplot(data=time_demand, x='time_index', y='mean', marker='o', ax=axes[0, 0])
axes[0, 0].set_title('Average demand by 15-minute slot')
axes[0, 0].set_xlabel('Time index: 0=00:00, 95=23:45')

hour_demand = train_clean.groupby('hour')[TARGET].mean().reset_index()
sns.barplot(data=hour_demand, x='hour', y=TARGET, ax=axes[0, 1])
axes[0, 1].set_title('Average demand by hour')

sns.boxplot(data=train_clean, x='day', y=TARGET, ax=axes[1, 0])
axes[1, 0].set_title('Demand by day')

sns.lineplot(data=train_clean, x='time_index', y=TARGET, hue='day', estimator='mean', errorbar=None, ax=axes[1, 1])
axes[1, 1].set_title('Demand by time slot and day')

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(22, 12))
cat_plot_cols = ['RoadType', 'Weather', 'LargeVehicles', 'Landmarks', 'NumberofLanes', 'is_peak_hour']

for ax, col in zip(axes.ravel(), cat_plot_cols):
    order = train_clean.groupby(col)[TARGET].mean().sort_values(ascending=False).index
    sns.barplot(data=train_clean, x=col, y=TARGET, order=order, estimator='mean', errorbar='ci', ax=ax)
    ax.set_title(f'Average demand by {col}')
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()


In [ ]:
geo_summary = train_clean.groupby('geohash').agg(
    rows=(TARGET, 'size'),
    mean_demand=(TARGET, 'mean'),
    median_demand=(TARGET, 'median'),
    lat=('geo_lat', 'first'),
    lon=('geo_lon', 'first')
).sort_values('mean_demand', ascending=False)

display(geo_summary.head(20))

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
sns.histplot(geo_summary['mean_demand'], bins=50, kde=True, ax=axes[0])
axes[0].set_title('Distribution of geohash-level mean demand')

plot_geo = geo_summary.reset_index()
scatter = axes[1].scatter(
    plot_geo['lon'], plot_geo['lat'],
    c=plot_geo['mean_demand'], s=np.clip(plot_geo['rows'], 5, 80), cmap='viridis', alpha=0.75
)
axes[1].set_title('Decoded geohash demand map')
axes[1].set_xlabel('Longitude')
axes[1].set_ylabel('Latitude')
plt.colorbar(scatter, ax=axes[1], label='Mean demand')
plt.tight_layout()
plt.show()


In [ ]:
numeric_cols_for_corr = [
    TARGET, 'day', 'hour', 'minute', 'time_index', 'total_minutes', 'NumberofLanes',
    'Temperature', 'temp_capped', 'geo_lat', 'geo_lon', 'large_vehicle_allowed',
    'has_landmark', 'is_peak_hour', 'is_morning_peak', 'is_evening_peak', 'is_late_night',
    'lane_vehicle_interaction', 'peak_lane_interaction'
]

corr = train_clean[numeric_cols_for_corr].corr(numeric_only=True)
plt.figure(figsize=(16, 12))
sns.heatmap(corr, cmap='coolwarm', center=0, linewidths=.5)
plt.title('Numeric feature correlation heatmap')
plt.show()

corr_to_target = corr[TARGET].drop(TARGET).sort_values(key=lambda s: s.abs(), ascending=False)
display(corr_to_target.to_frame('corr_with_demand'))


In [ ]:
compare_cols = ['hour', 'time_index', 'RoadType', 'Weather', 'NumberofLanes', 'LargeVehicles', 'Landmarks']
for col in compare_cols:
    print(f'\n===== Train/Test distribution: {col} =====')
    train_dist = train_clean[col].value_counts(normalize=True).rename('train_pct')
    test_dist = test_clean[col].value_counts(normalize=True).rename('test_pct')
    stability = pd.concat([train_dist, test_dist], axis=1).fillna(0)
    stability['train_pct'] = stability['train_pct'] * 100
    stability['test_pct'] = stability['test_pct'] * 100
    stability['abs_diff_pct'] = (stability['train_pct'] - stability['test_pct']).abs()
    display(stability.sort_values('abs_diff_pct', ascending=False).head(15).round(2))


## 6. Feature Engineering For Training

Create target-relevant features while keeping validation honest.

Important: target encoding is out-of-fold for train rows and train-fitted for test rows, so the model does not see its own target while learning.


In [ ]:
def add_frequency_features(train_df, test_df, cols):
    train = train_df.copy()
    test = test_df.copy()
    combined = pd.concat([train[cols], test[cols]], axis=0)
    total_rows = len(combined)
    for col in cols:
        counts = combined[col].value_counts(dropna=False)
        train[f'{col}_freq'] = train[col].map(counts).fillna(0) / total_rows
        test[f'{col}_freq'] = test[col].map(counts).fillna(0) / total_rows
    return train, test


def add_oof_target_encoding(train_df, test_df, cols, target=TARGET, group_col='day', smoothing=20):
    train = train_df.copy()
    test = test_df.copy()
    global_mean = train[target].mean()

    n_splits = min(5, train[group_col].nunique()) if group_col in train.columns else 5
    for col in cols:
        encoded = pd.Series(index=train.index, dtype=float)
        if n_splits >= 2:
            splitter = GroupKFold(n_splits=n_splits)
            split_iter = splitter.split(train, train[target], groups=train[group_col])
        else:
            from sklearn.model_selection import KFold
            splitter = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
            split_iter = splitter.split(train)

        for tr_idx, val_idx in split_iter:
            stats = train.iloc[tr_idx].groupby(col)[target].agg(['mean', 'count'])
            smooth = (stats['mean'] * stats['count'] + global_mean * smoothing) / (stats['count'] + smoothing)
            encoded.iloc[val_idx] = train.iloc[val_idx][col].map(smooth)

        full_stats = train.groupby(col)[target].agg(['mean', 'count'])
        full_smooth = (full_stats['mean'] * full_stats['count'] + global_mean * smoothing) / (full_stats['count'] + smoothing)
        train[f'{col}_target_mean_oof'] = encoded.fillna(global_mean)
        test[f'{col}_target_mean_oof'] = test[col].map(full_smooth).fillna(global_mean)
    return train, test


freq_cols = ['geohash', 'geohash_prefix_4', 'geohash_prefix_5', 'road_weather']
te_cols = ['geohash', 'geohash_prefix_4', 'RoadType', 'Weather', 'road_weather']

train_fe, test_fe = add_frequency_features(train_clean, test_clean, freq_cols)
train_fe, test_fe = add_oof_target_encoding(train_fe, test_fe, te_cols, group_col='day', smoothing=30)

train_fe['demand_log1p'] = np.log1p(train_fe[TARGET])

print('Feature engineered train shape:', train_fe.shape)
print('Feature engineered test shape :', test_fe.shape)
display(train_fe.head())


## 7. Encoding And Training-Ready Matrix

Use consistent label encoding for categorical variables. Tree-based models can use these directly; linear models should use one-hot encoding or scaling as needed.


In [ ]:
CATEGORICAL_FEATURES = [
    'geohash', 'geohash_prefix_3', 'geohash_prefix_4', 'geohash_prefix_5',
    'RoadType', 'LargeVehicles', 'Landmarks', 'Weather', 'road_weather'
]

NUMERIC_FEATURES = [
    'day', 'hour', 'minute', 'time_index', 'total_minutes',
    'hour_sin', 'hour_cos', 'time_index_sin', 'time_index_cos',
    'is_morning_peak', 'is_evening_peak', 'is_late_night', 'is_peak_hour',
    'NumberofLanes', 'has_many_lanes', 'Temperature', 'temp_capped',
    'RoadType_was_missing', 'Weather_was_missing', 'Temperature_was_missing', 'timestamp_invalid',
    'large_vehicle_allowed', 'has_landmark', 'lane_vehicle_interaction', 'peak_lane_interaction',
    'geo_lat', 'geo_lon',
    'geohash_freq', 'geohash_prefix_4_freq', 'geohash_prefix_5_freq', 'road_weather_freq',
    'geohash_target_mean_oof', 'geohash_prefix_4_target_mean_oof',
    'RoadType_target_mean_oof', 'Weather_target_mean_oof', 'road_weather_target_mean_oof'
]

FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

train_model = train_fe[[ID_COL, TARGET, 'demand_log1p'] + FEATURES].copy()
test_model = test_fe[[ID_COL] + FEATURES].copy()

label_encoders = {}
for col in CATEGORICAL_FEATURES:
    le = LabelEncoder()
    combined_values = pd.concat([train_model[col], test_model[col]], axis=0).astype(str).fillna('Unknown')
    le.fit(combined_values)
    train_model[col] = le.transform(train_model[col].astype(str).fillna('Unknown'))
    test_model[col] = le.transform(test_model[col].astype(str).fillna('Unknown'))
    label_encoders[col] = le

for col in FEATURES:
    train_model[col] = pd.to_numeric(train_model[col], errors='coerce')
    test_model[col] = pd.to_numeric(test_model[col], errors='coerce')
    median_value = train_model[col].median()
    train_model[col] = train_model[col].fillna(median_value)
    test_model[col] = test_model[col].fillna(median_value)

X = train_model[FEATURES]
y = train_model[TARGET]
y_log = train_model['demand_log1p']
X_test = test_model[FEATURES]

print('X shape:', X.shape)
print('y shape:', y.shape)
print('X_test shape:', X_test.shape)
print('Any missing in X:', int(X.isna().sum().sum()))
print('Any missing in X_test:', int(X_test.isna().sum().sum()))

display(X.head())


## 8. Feature Relevance Summary

A quick univariate relevance scan helps prioritize model features. This does not replace model-based importance, but it flags useful signal early.


In [ ]:
feature_relevance = X.join(y).corr(numeric_only=True)[TARGET].drop(TARGET)
feature_relevance = feature_relevance.reindex(feature_relevance.abs().sort_values(ascending=False).index)

display(feature_relevance.head(30).to_frame('corr_with_demand'))

plt.figure(figsize=(10, 8))
sns.barplot(x=feature_relevance.head(25).values, y=feature_relevance.head(25).index, palette='viridis')
plt.title('Top feature correlations with demand')
plt.xlabel('Correlation')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()


## 9. Save Training-Ready Data

Export processed matrices for modeling. `train_processed.csv` includes `demand`; `test_processed.csv` includes the same features and no target.


In [ ]:
train_output = train_model[[ID_COL, TARGET] + FEATURES]
test_output = test_model[[ID_COL] + FEATURES]

train_output.to_csv('train_processed.csv', index=False)
test_output.to_csv('test_processed.csv', index=False)

feature_metadata = pd.DataFrame({
    'feature': FEATURES,
    'type': ['numeric' if f in NUMERIC_FEATURES else 'categorical_label_encoded' for f in FEATURES]
})
feature_metadata.to_csv('feature_metadata.csv', index=False)

print('Saved train_processed.csv:', train_output.shape)
print('Saved test_processed.csv :', test_output.shape)
print('Saved feature_metadata.csv:', feature_metadata.shape)

display(feature_metadata)


## 10. Ready For Model Training

Use the variables below directly in later training cells:

- `X`: processed training features
- `y`: original demand target
- `y_log`: optional `log1p(demand)` target
- `X_test`: processed test features
- `FEATURES`: final feature list
- `CATEGORICAL_FEATURES`: label-encoded categorical feature names
- `NUMERIC_FEATURES`: numeric feature names


In [ ]:
print('Training matrix is ready.')
print(f'Number of features: {len(FEATURES)}')
print(f'Numeric features: {len(NUMERIC_FEATURES)}')
print(f'Categorical label-encoded features: {len(CATEGORICAL_FEATURES)}')
print('\nFirst 10 features:', FEATURES[:10])


## 11. High-Performance Model Training And Submission

The final submission is generated by `train_high_performance_model.py`. It trains LightGBM, XGBoost, and CatBoost, validates on early day 49, tunes ensemble weights, retrains on all available training rows, and writes `submission.csv`.


In [ ]:
import runpy

# This cell may take around 1-2 minutes depending on your machine.
runpy.run_path('model.py', run_name='__main__')

submission = pd.read_csv('submission.csv')
print(submission.shape)
display(submission.head())
